<a href="https://colab.research.google.com/github/MuhammadAzan30/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAzan30/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import subprocess, sys, os

REPO_URL = "https://github.com/MuhammadAzan30/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type:** Scoring (used to produce a ranking)

Each page gets a probability score (0-1) representing how likely it is to be
declining. Pages aren't just sorted into two hard buckets — the score itself
is the useful output, because it lets a content team rank every page and
pull the top N for review. This is why Notebook 01's pipeline reports
Precision@50 rather than plain accuracy — it's evaluating the ranking that
comes out of the score, not a single yes/no cutoff.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy:** `trend_direction == "down"` (binary: 1 if declining, 0 otherwise)

This is a proxy, not a perfect label — "declining" is derived from a trend
classification in the data, not a direct measurement of "this page truly
needs a refresh." A page could be flagged "down" for reasons unrelated to
content quality (seasonality, a SERP layout change), so the proxy captures
correlation with decline, not a guaranteed causal need for refresh.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric:** Precision@50

Of the top 50 pages the model ranks highest, what fraction are actually
declining? This matches the real action: a content team has limited
capacity and will only work through a fixed batch of pages per cycle, so
what matters is whether the *top* of the queue is right — not overall
accuracy across all 30,000 pages, most of which no one will ever look at.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one page, scored at a point in time.

In [8]:
# Sketch what the target column would actually look like
df["target_declining"] = (df["trend_direction"] == "down").astype(int)

print(df["target_declining"].value_counts())
print(f"\nBase rate: {df['target_declining'].mean():.1%} of pages are labeled declining")
df[["trend_direction", "target_declining"]].head(10)

target_declining
1    16262
0    13738
Name: count, dtype: int64

Base rate: 54.2% of pages are labeled declining


,trend_direction,target_declining
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


In [9]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Show the unit of analysis as an actual dataframe
print(f"Shape: {df.shape[0]} rows (pages), {df.shape[1]} columns")
df.head(5)

Shape: 30000 rows (pages), 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Notebook 01 already showed this empirically: a hand-written rule got 24%
Precision@50, while a random forest got 74% — roughly 3x better. A fixed
rule can only threshold on one or two signals at a time (e.g. "flag if
position > 20"), but decline risk depends on *interactions* between
signals — position, CTR, content age, word count together — that a simple
rule can't capture. A learned model finds those interactions instead of
requiring a human to hand-specify every combination.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.